In [1]:
# ── TF 2.20 wheels (Kaggle only) ───────────────────────────────────────────
import os

KAGGLE = os.path.exists('/kaggle')
TRAIN_MODE = not KAGGLE  # False = inference / submission only

"""if KAGGLE:
    import subprocess
    subprocess.run(['pip', 'install', '-q', '--no-deps',
        '/kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl',
        '/kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorboard-2.20.0-py3-none-any.whl',
    ], check=True)
    # subprocess.run(['pip', 'install', 'tf2onnx', 'onnxruntime', 'onnxscript'], check=True)"""

"if KAGGLE:\n    import subprocess\n    subprocess.run(['pip', 'install', '-q', '--no-deps',\n        '/kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl',\n        '/kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorboard-2.20.0-py3-none-any.whl',\n    ], check=True)\n    # subprocess.run(['pip', 'install', 'tf2onnx', 'onnxruntime', 'onnxscript'], check=True)"

In [2]:
import os, random, logging, time, warnings
from contextlib import contextmanager
from pathlib import Path

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import h5py
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')
import torch
import torch.nn as nn
import torchvision.models as tv_models
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

print('TF:', tf.__version__, '  PyTorch:', torch.__version__)

E0000 00:00:1776980170.753803      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776980170.808792      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776980171.264073      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776980171.264116      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776980171.264119      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776980171.264121      23 computation_placer.cc:177] computation placer already registered. Please check linka

TF: 2.19.0   PyTorch: 2.10.0+cu128


## Config

In [3]:
# ── Paths ───────────────────────────────────────────────────────────────────
if KAGGLE:
    COMP_DIR         = Path('/kaggle/input/competitions/birdclef-2026')
    MODEL_DIR        = Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1')
    CKPT_DIR         = Path('/kaggle/working/checkpoints')
    HDF5_DIR         = Path('/kaggle/working/hdf5')
    OUTPUT_DIR       = Path('/kaggle/working')
    CKPT_PATH        = Path('/kaggle/input/datasets/benotdupire/birdclef2026-checkpoints/fold0_best.pt')
    RESNET_CKPT_PATH = Path('/kaggle/input/datasets/benotdupire/birdclef2026-checkpoints/resnet_best3.pt')
else:
    ROOT        = (Path.home() / 'Documents/programming/birdclef+2026/birdclef-2026').resolve()
    COMP_DIR    = ROOT
    MODEL_DIR   = None   # kagglehub will download
    WORKING_DIR = ROOT / 'working'
    CKPT_DIR    = WORKING_DIR / 'checkpoints'
    HDF5_DIR    = WORKING_DIR / 'hdf5'
    OUTPUT_DIR  = WORKING_DIR / 'outputs'
    CKPT_PATH   = CKPT_DIR / 'fold0_best.pt'

TRAIN_AUDIO  = COMP_DIR / 'train_audio'
TEST_SND_DIR = COMP_DIR / 'test_soundscapes'
TRAIN_SND    = COMP_DIR / 'train_soundscapes'
META_CSV     = COMP_DIR / 'train.csv'
SAMPLE_SUB   = COMP_DIR / 'sample_submission.csv'
HDF5_PATH    = HDF5_DIR / 'train_emb.h5'
OUT_PATH     = OUTPUT_DIR / 'submission.csv'

for d in [CKPT_DIR, HDF5_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if not TEST_SND_DIR.exists() or not any(TEST_SND_DIR.glob('*')):
    TEST_SND_DIR = TRAIN_SND
    print('test_soundscapes empty — using train_soundscapes')

# ── Audio ───────────────────────────────────────────────────────────────────
SAMPLE_RATE   = 32_000
CLIP_DURATION = 5
CLIP_SAMPLES  = SAMPLE_RATE * CLIP_DURATION

# ── Perch ───────────────────────────────────────────────────────────────────
PERCH_EMBED_DIM = 1536

# ── Training ────────────────────────────────────────────────────────────────
SEED            = 42
NUM_FOLDS       = 5
FOLD            = 0
TRAIN_EPOCHS    = 30
BATCH_SIZE      = 256
ACCUM_STEPS     = 1
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
WARMUP_EPOCHS   = 2
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA     = 0.4
NUM_WORKERS     = 4
PIN_MEMORY      = True
DROP_RATE       = 0.3

# ── Inference ───────────────────────────────────────────────────────────────
INFER_BATCH_SIZE = 64
AMP              = False
DEVICE           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [4]:
# ── Spectrogram config ────────────────────────────────────────────────────
N_FFT        = 1024
HOP_LENGTH   = 320          # 32000 / 320 = 100 frames/sec
N_MELS       = 128
F_MIN        = 50
F_MAX        = 14_000
SPEC_WIDTH   = 500          # frames pour 5 sec @ hop=320  (32000*5/320 ≈ 500)
IMG_MEAN     = 0.5
IMG_STD      = 0.5

def audio_to_melspec(wav: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    """
    wav  : float32 mono array  → zéro-paddé à CLIP_SAMPLES si nécessaire
    return: float32 numpy (N_MELS, SPEC_WIDTH), normalisé [0,1] en dB
    """
    if len(wav) < CLIP_SAMPLES:
        wav = np.pad(wav, (0, CLIP_SAMPLES - len(wav)))
    else:
        wav = wav[:CLIP_SAMPLES]

    mel = librosa.feature.melspectrogram(
        y=wav.astype(np.float32),
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=F_MIN,
        fmax=F_MAX,
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)   # (N_MELS, T)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

    # Crop ou pad en largeur à SPEC_WIDTH
    T = mel_db.shape[1]
    if T >= SPEC_WIDTH:
        mel_db = mel_db[:, :SPEC_WIDTH]
    else:
        mel_db = np.pad(mel_db, ((0, 0), (0, SPEC_WIDTH - T)))

    return mel_db.astype(np.float32)   # (N_MELS, SPEC_WIDTH)

In [5]:
class MelSpecDataset(Dataset):
    """
    Lit les fichiers audio .ogg et retourne (spectrogram_tensor, label_vector).
    Utilisé pour l'entraînement du ResNet — en parallèle du pipeline Perch/MLP.
    """
    def __init__(self, df: pd.DataFrame, s2i: dict, augment: bool = False):
        self.df      = df.reset_index(drop=True)
        self.s2i     = s2i
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def _load_clip(self, path) -> np.ndarray:
        try:
            wav, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=CLIP_DURATION)
        except Exception:
            wav = np.zeros(CLIP_SAMPLES, dtype=np.float32)
        return wav.astype(np.float32)

    def _augment_wav(self, wav: np.ndarray) -> np.ndarray:
        # Gain aléatoire
        wav = wav * np.random.uniform(0.8, 1.2)
        # Décalage temporel aléatoire
        shift = np.random.randint(0, SAMPLE_RATE)
        wav = np.roll(wav, shift)
        # Bruit gaussien léger
        if np.random.rand() < 0.5:
            wav = wav + np.random.randn(*wav.shape).astype(np.float32) * 0.005
        return wav

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = TRAIN_AUDIO / row['filename']
        wav  = self._load_clip(path)

        if self.augment:
            wav = self._augment_wav(wav)
        wav = wav / (np.abs(wav).max() + 1e-6)   # normalisation

        spec = audio_to_melspec(wav)              # (N_MELS, SPEC_WIDTH)

        # Random crop en width si augment
        if self.augment:
            T = spec.shape[1]
            max_start = max(0, T - SPEC_WIDTH)
            start = random.randint(0, max_start) if max_start > 0 else 0
            spec = spec[:, start:start + SPEC_WIDTH]
            if spec.shape[1] < SPEC_WIDTH:
                spec = np.pad(spec, ((0,0),(0, SPEC_WIDTH - spec.shape[1])))

        # (1, H, W) pour conv2d mono-canal + normalisation ImageNet-style
        spec_t = torch.from_numpy(spec).unsqueeze(0)
        spec_t = (spec_t - IMG_MEAN) / IMG_STD

        # Label multi-hot
        label = np.zeros(len(self.s2i), dtype=np.float32)
        if row['primary_label'] in self.s2i:
            label[self.s2i[row['primary_label']]] = 1.0
        raw = row.get('secondary_labels', '')
        if pd.notna(raw) and isinstance(raw, str) and raw.strip() not in ('', '[]'):
            for sl in raw.strip('[]').split(','):
                sl = sl.strip().strip("'\" ")
                if sl in self.s2i:
                    label[self.s2i[sl]] = 1.0

        return spec_t, torch.from_numpy(label)

In [6]:
class ResNetSpecHead(nn.Module):
    def __init__(self, n_classes, backbone='resnet18', drop_rate=DROP_RATE):
        super().__init__()
        if backbone == 'resnet18':
            base = tv_models.resnet18(weights=None)
        else:
            base = tv_models.resnet50(weights=None)
        in_features   = base.fc.in_features
        base.fc       = nn.Identity()
        self.backbone = base
        self.head     = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, n_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))

In [7]:
@torch.no_grad()
def infer_resnet_soundscape(model: ResNetSpecHead, wav: np.ndarray) -> dict:
    """
    Découpe un wav (1 min) en fenêtres de 5 sec.
    Retourne {end_time: probs_array (n_classes,)}.
    """
    model.eval()
    results = {}
    n_steps = len(wav) // CLIP_SAMPLES

    for i in range(n_steps):
        clip = wav[i * CLIP_SAMPLES:(i + 1) * CLIP_SAMPLES]
        clip = clip / (np.abs(clip).max() + 1e-6)

        spec = audio_to_melspec(clip)              # (N_MELS, SPEC_WIDTH)
        spec_t = torch.from_numpy(spec).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
        spec_t = (spec_t - IMG_MEAN) / IMG_STD

        probs = torch.sigmoid(model(spec_t.to(DEVICE))).squeeze().cpu().numpy()
        results[(i + 1) * CLIP_DURATION] = probs

    return results


def ensemble_perch_resnet(perch_probs: np.ndarray, resnet_probs: np.ndarray,
                          alpha: float = 0.4) -> np.ndarray:
    """
    Moyenne pondérée : (1-alpha)*Perch + alpha*ResNet
    alpha=0.4 donne légèrement plus de poids à Perch (modèle plus puissant).
    """
    return (1.0 - alpha) * perch_probs + alpha * resnet_probs

In [8]:
def get_logger(name='birdclef'):
    logger = logging.getLogger(name)
    if not logger.handlers:
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter('[%(asctime)s] %(levelname)s - %(message)s', '%H:%M:%S'))
        logger.addHandler(h)
        logger.setLevel(logging.INFO)
    return logger

logger = get_logger()


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def build_label_map(meta_df):
    species = sorted(meta_df['primary_label'].unique().tolist())
    s2i = {s: i for i, s in enumerate(species)}
    i2s = {i: s for s, i in s2i.items()}
    return s2i, i2s


def encode_labels(primary, secondary, s2i):
    vec = np.zeros(len(s2i), dtype=np.float32)
    if primary in s2i:
        vec[s2i[primary]] = 1.0
    for lbl in (secondary or []):
        if lbl in s2i:
            vec[s2i[lbl]] = 1.0
    return vec


def competition_score(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average='macro'), keep


def mixup_data(x, y, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def save_checkpoint(state, path):
    torch.save(state, path)
    logger.info(f'Saved → {path}')


@contextmanager
def autocast_ctx():
    if AMP and torch.cuda.is_available():
        with torch.amp.autocast('cuda'):
            yield
    else:
        yield


def get_scaler():
    return torch.amp.GradScaler('cuda') if AMP and torch.cuda.is_available() else None

In [9]:
def load_wave(path):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        wave, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return wave.astype(np.float32)


def pad_or_trim(wave, length=CLIP_SAMPLES):
    if len(wave) < length:
        return np.pad(wave, (0, length - len(wave)))
    start = np.random.randint(0, len(wave) - length + 1)
    return wave[start : start + length]


def center_crop(wave, length=CLIP_SAMPLES):
    if len(wave) <= length:
        return np.pad(wave, (0, length - len(wave)))
    start = (len(wave) - length) // 2
    return wave[start : start + length]


def normalize_wave(wave):
    return wave / (np.abs(wave).max() + 1e-6)


def augment_wave(wave, p=0.5):
    if np.random.rand() < p:
        wave = wave + np.random.randn(*wave.shape).astype(np.float32) * 0.005
    if np.random.rand() < p:
        wave = np.roll(wave, int(np.random.uniform(-0.2, 0.2) * len(wave)))
    if np.random.rand() < p * 0.5:
        wave = wave * (10 ** (np.random.uniform(-6, 6) / 20))
    return wave


def chunk_wave(wave, window_samples=CLIP_SAMPLES):
    start, t = 0, CLIP_DURATION
    while start < len(wave):
        chunk = wave[start : start + window_samples]
        if len(chunk) < window_samples:
            chunk = np.pad(chunk, (0, window_samples - len(chunk)))
        yield chunk, t
        start += window_samples
        t     += CLIP_DURATION

In [10]:
class BirdDataset(Dataset):
    def __init__(self, df, s2i, mode='hdf5', augment=False, embedder=None):
        self.df       = df.reset_index(drop=True)
        self.s2i      = s2i
        self.mode     = mode
        self.augment  = augment
        self.embedder = embedder

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.mode == 'hdf5':
            label_key = str(row['primary_label'])
            file_key = os.path.splitext(row['filename'].split('/')[-1])[0] + '.ogg'
            with h5py.File(HDF5_PATH, 'r') as f:
                emb = f[label_key][file_key][:]
            if emb.ndim > 1:
                emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
        else:
            wave = load_wave(TRAIN_AUDIO / row['filename'])
            wave = pad_or_trim(wave) if self.augment else center_crop(wave)
            if self.augment:
                wave = augment_wave(wave)
            wave = normalize_wave(wave)
            emb  = self.embedder.embed(wave)

        raw = row.get('secondary_labels', None)
        secondary = []
        if pd.notna(raw) and isinstance(raw, str) and raw.strip() not in ('', '[]'):
            secondary = [s.strip().strip("'\"") for s in raw.strip('[]').split(',') if s.strip()]

        return (
            torch.from_numpy(emb.astype(np.float32)),
            torch.from_numpy(encode_labels(row['primary_label'], secondary, self.s2i))
        )


def get_fold_dfs(meta, fold=FOLD, n_folds=NUM_FOLDS, seed=SEED):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for f, (tr, val) in enumerate(skf.split(meta, meta['primary_label'])):
        if f == fold:
            return meta.iloc[tr].copy(), meta.iloc[val].copy()
    raise ValueError(f'Fold {fold} not found')


def get_loaders(meta, s2i, fold=FOLD, mode='hdf5', embedder=None):
    train_df, val_df = get_fold_dfs(meta, fold=fold)
    train_loader = DataLoader(
        BirdDataset(train_df, s2i, mode=mode, augment=True, embedder=embedder),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        pin_memory=False, drop_last=True,
    )
    val_loader = DataLoader(
        BirdDataset(val_df, s2i, mode=mode, augment=False, embedder=embedder),
        batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0,
        pin_memory=False, drop_last=False,
    )
    return train_loader, val_loader

In [11]:
# ── ResNet sur spectrogrammes ─────────────────────────────────────────────
RESNET_EPOCHS    = 30
RESNET_LR        = 1e-3
RESNET_BATCH     = 64  # plus petit car images

import torchvision.transforms as T

# ── Mel spectrogram transform ─────────────────────────────────────────────
class MelSpecDataset(Dataset):
    def __init__(self, df, s2i, audio_dir, augment=False):
        self.df            = df.reset_index(drop=True)
        self.s2i           = s2i
        self.audio_dir     = Path(audio_dir)
        self.augment       = augment
        self.mel_cache_dir = Path('/kaggle/working/mel_cache')
        self.mel_cache_dir.mkdir(exist_ok=True)

    def __len__(self):
        return len(self.df)

    def _spec_augment(self, mel):
        """Masquage aléatoire fréquence + temps."""
        _, n_mels, n_frames = mel.shape
        if self.augment:
            f_mask = random.randint(0, 20)
            f_start = random.randint(0, n_mels - f_mask)
            mel[:, f_start:f_start + f_mask, :] = 0
            t_mask = random.randint(0, 40)
            t_start = random.randint(0, n_frames - t_mask)
            mel[:, :, t_start:t_start + t_mask] = 0
        return mel


    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = np.zeros(len(self.s2i), dtype=np.float32)
        if row['primary_label'] in self.s2i:
            label[self.s2i[row['primary_label']]] = 1.0
        raw = row.get('secondary_labels', '')
        if pd.notna(raw) and isinstance(raw, str) and raw.strip() not in ('', '[]'):
            for sl in raw.strip('[]').split(','):
                sl = sl.strip().strip("'\" ")
                if sl in self.s2i:
                    label[self.s2i[sl]] = 1.0

    
        cache_path = self.mel_cache_dir / (row['filename'].replace('/', '_') + '.npy')
    
        if cache_path.exists():
            mel = np.load(cache_path)
        else:
            path = self.audio_dir / row['filename']
            try:
                wave, sr = sf.read(str(path), dtype='float32')
            except Exception:
                wave, sr = np.zeros(CLIP_SAMPLES, dtype=np.float32), SAMPLE_RATE
    
            if wave.ndim > 1:
                wave = wave.mean(axis=1)
            if len(wave) < CLIP_SAMPLES:
                wave = np.pad(wave, (0, CLIP_SAMPLES - len(wave)))
            else:
                wave = wave[:CLIP_SAMPLES]
            if sr != SAMPLE_RATE:
                wave = librosa.resample(wave, orig_sr=sr, target_sr=SAMPLE_RATE)
    
            mel = librosa.feature.melspectrogram(
                y=wave, sr=SAMPLE_RATE,
                n_fft=N_FFT, hop_length=HOP_LENGTH,
                n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX
            )
            mel = librosa.power_to_db(mel, ref=np.max)
            mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
            np.save(cache_path, mel)
    
        mel = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).expand(3, -1, -1).clone()
        mel = T.functional.resize(mel, [128, 256])
        mel = T.functional.normalize(mel, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        mel = self._spec_augment(mel)
        return mel, label


# ── Inférence ResNet sur un clip audio brut ───────────────────────────────
@torch.no_grad()
def infer_resnet_soundscape(model, wave_clip):
    """wave_clip : np.array float32 de longueur CLIP_SAMPLES.
    Retourne dict {5: probs_array} pour compatibilité avec le loop d'inférence."""
    mel = librosa.feature.melspectrogram(
        y=wave_clip, sr=SAMPLE_RATE,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX
    )
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
    mel = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).expand(3, -1, -1)
    mel = T.functional.resize(mel, [128, 256])
    mel = T.functional.normalize(mel, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    mel = mel.unsqueeze(0).to(DEVICE)
    probs = torch.sigmoid(model(mel)).cpu().numpy()[0]
    return {5: probs}


# ── Ensemble Perch + ResNet ───────────────────────────────────────────────
def ensemble_perch_resnet(perch_probs, resnet_probs, alpha=0.4):
    """alpha = poids ResNet, (1-alpha) = poids Perch."""
    return (1 - alpha) * perch_probs + alpha * resnet_probs


meta_df = pd.read_csv(META_CSV)
s2i, i2s = build_label_map(meta_df)


# ── Fonction d'entraînement ResNet ───────────────────────────────────────
def train_resnet(meta_df, s2i, n_epochs=30, batch_size=64, lr=1e-3, backbone='resnet18', audio_dir=None, fold=FOLD):
    if audio_dir is None:
        audio_dir = TRAIN_AUDIO

    num_classes = len(s2i)
    df = meta_df[meta_df['primary_label'].isin(s2i)].reset_index(drop=True)

    skf    = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
    splits = list(skf.split(df, df['primary_label']))
    tr_idx, val_idx = splits[fold]

    train_ds = MelSpecDataset(df.iloc[tr_idx], s2i, audio_dir, augment=True)
    val_ds   = MelSpecDataset(df.iloc[val_idx], s2i, audio_dir, augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model     = ResNetSpecHead(n_classes=num_classes, backbone=backbone).to(DEVICE)
    criterion = BCEWithLabelSmoothing(
    pos_weight=None,
    smoothing=LABEL_SMOOTHING
    )
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = OneCycleLR(optimizer, max_lr=lr,
                           steps_per_epoch=len(train_loader),
                           epochs=n_epochs, pct_start=WARMUP_EPOCHS/n_epochs)
    scaler    = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_auc = 0.0
    logger.info(f'ResNet — train={len(train_ds)}  val={len(val_ds)}  classes={num_classes}')

    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss = 0.0
        for mels, labels in tqdm(train_loader, desc=f'  [ResNet] Epoch {epoch}', leave=False):
            mels, labels = mels.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                loss = criterion(model(mels), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
        tr_loss = total_loss / len(train_loader)

        model.eval()
        all_probs, all_labels = [], []
        val_loss = 0.0
        with torch.no_grad():
            for mels, labels in val_loader:
                mels, labels = mels.to(DEVICE), labels.to(DEVICE)
                logits = model(mels)
                val_loss += criterion(logits, labels).item()
                all_probs.append(torch.sigmoid(logits).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        val_loss /= len(val_loader)
        all_probs  = np.concatenate(all_probs)        
        all_labels = np.vstack(all_labels)            
        
        keep = all_labels.sum(axis=0) > 0
        try:
            auc = roc_auc_score(all_labels[:, keep], all_probs[:, keep], average='macro')
        except ValueError:
            auc = 0.0

        logger.info(f'  [ResNet] Epoch {epoch}/{n_epochs}  tr={tr_loss:.4f}  val={val_loss:.4f}  auc={auc:.4f}  lr={scheduler.get_last_lr()[0]:.2e}')

        if auc > best_auc:
            best_auc = auc
            torch.save({'model': model.state_dict(), 'epoch': epoch, 'val_auc': best_auc},
                       RESNET_CKPT_PATH)
            logger.info(f'  [ResNet] Saved → {RESNET_CKPT_PATH}')

    logger.info(f'ResNet best AUC: {best_auc:.4f}')
    return model

In [12]:
class BCEWithLabelSmoothing(nn.Module):
    def __init__(self, pos_weight, smoothing=LABEL_SMOOTHING):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        return self.bce(logits, targets * (1 - self.smoothing) + self.smoothing / 2)

In [13]:
if TRAIN_MODE:
    resnet_model = train_resnet(
        meta_df   = meta_df,  
        s2i       = s2i,
        n_epochs  = 30,
        batch_size= 32,
        lr        = 1e-3,
        backbone  = 'resnet18',
        fold      = FOLD,
    )

## Inference

In [14]:
# ── Inférence ResNet uniquement ───────────────────────────────────────────
meta = pd.read_csv(META_CSV)
s2i, i2s = build_label_map(meta)
counts = meta['primary_label'].value_counts()
valid_species = counts[counts >= 5].index
meta_filtered = meta[meta['primary_label'].isin(valid_species)].reset_index(drop=True)
s2i_filtered, i2s_filtered = build_label_map(meta_filtered)



import time
import torchaudio.transforms as AT
import torchvision.transforms as T

ss            = pd.read_csv(SAMPLE_SUB)
sub_species   = [c for c in ss.columns if c != 'row_id']
uniform_prior = 1.0 / len(sub_species)

# Mapping ResNet (192 classes) → colonnes submission
sub_col_idx  = {sp: i for i, sp in enumerate(sub_species)}
train_to_sub = {ri: sub_col_idx[sp] for ri, sp in i2s.items() if sp in sub_col_idx}
logger.info(f'Mapped: {len(train_to_sub)}/{len(i2s)}')

# Charger le ResNet
resnet_infer = ResNetSpecHead(n_classes=len(s2i), backbone='resnet18').to(DEVICE)
resnet_ckpt  = torch.load(RESNET_CKPT_PATH, map_location='cpu', weights_only=False)
resnet_infer.load_state_dict(resnet_ckpt['model'])
resnet_infer.eval()
logger.info(f'ResNet loaded — val_auc={resnet_ckpt.get("val_auc", 0):.4f}')

# ── Transforms GPU ────────────────────────────────────────────────────────
mel_transform   = AT.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX,
).to(DEVICE)
amplitude_to_db = AT.AmplitudeToDB().to(DEVICE)

@torch.no_grad()
def infer_resnet_batch(model, clips):
    """clips : np.array shape (N, CLIP_SAMPLES)"""
    x   = torch.tensor(clips, dtype=torch.float32).to(DEVICE)
    mel = mel_transform(x)
    mel = amplitude_to_db(mel)
    mel_min = mel.flatten(1).min(dim=1).values[:, None, None]
    mel_max = mel.flatten(1).max(dim=1).values[:, None, None]
    mel     = (mel - mel_min) / (mel_max - mel_min + 1e-6)
    mel     = mel.unsqueeze(1).expand(-1, 3, -1, -1)
    mel     = T.functional.resize(mel, [128, 256])
    mel     = T.functional.normalize(mel, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    return torch.sigmoid(model(mel)).cpu().numpy()

# Découvrir les soundscapes
stems = ss['row_id'].str.rsplit('_', n=1).str[0].unique()
soundscape_paths = []
for stem in stems:
    for ext in ('.ogg', '.wav', '.flac'):
        p = TEST_SND_DIR / (stem + ext)
        if p.exists():
            soundscape_paths.append(p)
            break
if not soundscape_paths:
    soundscape_paths = sorted(TEST_SND_DIR.glob('*.ogg')) + sorted(TEST_SND_DIR.glob('*.wav'))
logger.info(f'Soundscapes: {len(soundscape_paths)}')

# ── Boucle d'inférence ────────────────────────────────────────────────────
rows = []
t0   = time.perf_counter()

for path in tqdm(soundscape_paths, desc='Inference'):
    wave, sr = sf.read(str(path), dtype='float32')
    if wave.ndim > 1:
        wave = wave.mean(axis=1)
    if len(wave) < 12 * CLIP_SAMPLES:
        wave = np.pad(wave, (0, 12 * CLIP_SAMPLES - len(wave)))
    else:
        wave = wave[:12 * CLIP_SAMPLES]

    clips       = wave.reshape(12, CLIP_SAMPLES)
    probs_batch = infer_resnet_batch(resnet_infer, clips)

    for i, probs in enumerate(probs_batch):
        end_t = (i + 1) * 5
        out   = np.full(len(sub_species), uniform_prior, dtype=np.float32)
        for ri, si in train_to_sub.items():
            out[si] = probs[ri]
        rows.append({'row_id': f'{path.stem}_{end_t}', **dict(zip(sub_species, out.tolist()))})

logger.info(f'Inference done in {time.perf_counter() - t0:.1f}s')

# ── Sauvegarde submission ─────────────────────────────────────────────────
if not rows:
    logger.warning('No rows — filling with uniform prior')
    sub_final = ss.copy()
    for sp in sub_species:
        sub_final[sp] = uniform_prior
else:
    sub_pred  = pd.DataFrame(rows)
    sub_final = ss[['row_id']].merge(sub_pred, on='row_id', how='left')
    for sp in sub_species:
        sub_final[sp] = sub_final[sp].fillna(uniform_prior)

sub_final.to_csv(OUT_PATH, index=False)
logger.info(f'Saved → {OUT_PATH}  shape={sub_final.shape}')
sub_final.head(3)

[21:36:47] INFO - Mapped: 206/206
[21:36:49] INFO - ResNet loaded — val_auc=0.9124
[21:36:49] INFO - Soundscapes: 0
Inference: 0it [00:00, ?it/s]
[21:36:49] INFO - Inference done in 0.0s
[21:36:49] WARNING - No rows — filling with uniform prior
[21:36:49] INFO - Saved → /kaggle/working/submission.csv  shape=(3, 235)


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
